In [1]:
# ============================================================
# CELL 1: IMPORTS & CONFIG
# Novel Intra-Region LORO Cross-Validation
#
# Key idea: Instead of training across ALL regions and testing
# on one held-out region (classic LORO), we ask a harder
# question: does the EWS model generalize WITHIN a region?
#
# For each macro-region (NorCal, MidCal, SoCal, BigSur),
# we run LORO using ONLY the sites within that region:
#   - Train on (N-1) sites within the same region
#   - Test on the 1 held-out site
#
# This is a stricter test than cross-region LORO because:
#   1. Training set is smaller (fewer sites per region)
#   2. Sites within a region share similar oceanographic
#      forcing, so the model cannot "cheat" using region-
#      level differences as a shortcut.
#   3. If it works, EWS generalizes within a coherent ocean
#      regime — a genuinely novel validation design.
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from scipy.stats import mannwhitneyu, kruskal
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────
BASE    = Path('../../1_DATA/processed')
NC_PATH = Path('/Users/tonylin/Documents/kelp_project/1_DATA/raw/LandsatKelpBiomass_2025_Q3_v2_withmetadata.nc')
FIG_DIR = Path('../../5_FIGURES/loro_intraregion')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Model / EWS constants ─────────────────────────────────────
FEATURES    = ['ews_composite', 'heat_lag4', 'upwelling', 'heat_x_ews']
TARGET      = 'onset'
ROLL_WIN    = 20
HEAT_LAG    = 4
THRESHOLD   = 0.35
WARN_WINDOW = 4
BOOTSTRAP_B = 2000

# ── Palette ───────────────────────────────────────────────────
# Ocean-science colour scheme: cool blues/teals for north,
# warm coral for central, muted sage for south, violet for Big Sur
REGION_META = {
    'norcal': dict(color='#2196a8', label='Northern CA',   lat_range=(38.0, 42.0)),
    'midcal': dict(color='#d4622a', label='Central CA',    lat_range=(36.5, 38.5)),
    'bigsur': dict(color='#7b5ea7', label='Big Sur',       lat_range=(34.8, 36.5)),
    'socal':  dict(color='#3a9e6e', label='Southern CA',   lat_range=(32.5, 34.8)),
}

# Central CA upwelling core (a priori from oceanographic literature)
CENTRAL_LAT_MIN = 36.5
CENTRAL_LAT_MAX = 38.5

# ── Site catalogue: (name, lat_mid, lon_mid, region) ──────────
# Ordered north → south; must match bboxes in stress-test v2
ALL_SITES = [
    ('Crescent City',    41.75, -124.0, 'norcal'),
    ('Cape Mendocino',   40.45, -124.3, 'norcal'),
    ('Bodega Bay',       38.40, -123.2, 'midcal'),
    ('Point Reyes',      38.00, -122.9, 'midcal'),
    ('Half Moon Bay',    37.50, -122.5, 'midcal'),
    ('Santa Cruz',       37.00, -122.1, 'midcal'),
    ('Point Sur',        36.40, -121.7, 'bigsur'),
    ('Cambria',          35.55, -121.0, 'bigsur'),
    ('Morro Bay',        35.35, -120.8, 'bigsur'),
    ('Point Conception', 34.50, -120.5, 'socal'),
    ('Santa Barbara',    34.35, -119.8, 'socal'),
    ('Ventura',          34.25, -119.2, 'socal'),
    ('Palos Verdes',     33.75, -118.3, 'socal'),
    ('Laguna Beach',     33.55, -117.7, 'socal'),
    ('San Diego',        32.80, -117.2, 'socal'),
]
SITES_DF = pd.DataFrame(ALL_SITES,
    columns=['name', 'lat', 'lon', 'region'])
SITES_DF['central'] = SITES_DF['lat'].between(
    CENTRAL_LAT_MIN, CENTRAL_LAT_MAX)

# matplotlib global style
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.labelsize':   11,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.linewidth':   1.3,
    'figure.dpi':       150,
    'xtick.major.width':1.2,
    'ytick.major.width':1.2,
})

print('✓ Config loaded')
print(f'  Total sites: {len(SITES_DF)}')
print(f'  Sites per region:')
for r, g in SITES_DF.groupby('region'):
    print(f'    {REGION_META[r]["label"]:15s}: {len(g)} sites')

✓ Config loaded
  Total sites: 15
  Sites per region:
    Big Sur        : 3 sites
    Central CA     : 4 sites
    Northern CA    : 2 sites
    Southern CA    : 6 sites


In [2]:
# ============================================================
# CELL 2: DATA LOADING
# Tries processed CSVs first; falls back to raw NetCDF.
# ============================================================
import xarray as xr
import re
import time

# ── SST (NOAA OISST) ──────────────────────────────────────────
print('Loading SST...')
try:
    sst_url = 'https://psl.noaa.gov/thredds/dodsC/Datasets/noaa.oisst.v2.highres/sst.mon.mean.nc'
    ds_sst  = xr.open_dataset(sst_url)
    lat_nm  = 'lat'  if 'lat'  in ds_sst.coords else 'latitude'
    lon_nm  = 'lon'  if 'lon'  in ds_sst.coords else 'longitude'
    sst_ca  = ds_sst['sst'].sel({
        lat_nm: slice(32, 43),
        lon_nm: slice(234, 244)   # 360° coords: 234=−126°, 244=−116°
    })
    print('  ✓ SST loaded (OPeNDAP)')
except Exception as e:
    print(f'  ✗ SST load failed: {e}')
    sst_ca = None

# ── Upwelling (NOAA ERDDAP) ───────────────────────────────────
def get_ui(dataset_id):
    url = (f'https://coastwatch.pfeg.noaa.gov/erddap/griddap/{dataset_id}.csvp'
           f'?upwelling_index,upwelling_index_anomaly')
    df  = pd.read_csv(url)
    df.columns = [re.sub(r'\s*\(.*\)$', '', c).strip() for c in df.columns]
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time').sort_index()
    df.index = df.index.tz_localize(None)
    return df.rename(columns={'upwelling_index':'ui',
                              'upwelling_index_anomaly':'ui_anom'})

print('Loading upwelling...')
try:
    ui_cache = {}
    for lat_code in [33, 36, 39, 42]:
        try:
            ui_cache[lat_code] = get_ui(f'erdUI{lat_code}mo')
            print(f'  ✓ UI{lat_code}')
        except Exception:
            print(f'  ✗ UI{lat_code} unavailable')
    if 42 not in ui_cache and 39 in ui_cache:
        ui_cache[42] = ui_cache[39]
except Exception as e:
    print(f'  ✗ upwelling: {e}')
    ui_cache = {}

# ── Kelp (NetCDF) ─────────────────────────────────────────────
SITE_BBOXES = {
    'Crescent City':    (41.5,  42.0, -124.5, -123.5),
    'Cape Mendocino':   (40.2,  40.7, -124.8, -123.8),
    'Bodega Bay':       (38.2,  38.6, -123.5, -122.8),
    'Point Reyes':      (37.8,  38.2, -123.2, -122.5),
    'Half Moon Bay':    (37.3,  37.7, -122.8, -122.1),
    'Santa Cruz':       (36.8,  37.2, -122.4, -121.8),
    'Point Sur':        (36.2,  36.6, -122.0, -121.4),
    'Cambria':          (35.4,  35.8, -121.3, -120.7),
    'Morro Bay':        (35.2,  35.5, -121.0, -120.5),
    'Point Conception': (34.3,  34.7, -120.8, -120.1),
    'Santa Barbara':    (34.2,  34.5, -120.1, -119.5),
    'Ventura':          (34.1,  34.4, -119.5, -118.9),
    'Palos Verdes':     (33.6,  33.9, -118.6, -118.0),
    'Laguna Beach':     (33.4,  33.7, -117.9, -117.4),
    'San Diego':        (32.6,  33.0, -117.5, -116.9),
}

raw_kelp = {}
if NC_PATH.exists():
    from netCDF4 import Dataset, num2date
    print('Extracting kelp from NetCDF...')
    with Dataset(NC_PATH, 'r') as nc:
        lat_arr = nc.variables['latitude'][:]
        lon_arr = nc.variables['longitude'][:]
        tvar    = nc.variables['time']
        tvals   = num2date(tvar[:], units=tvar.units,
                           calendar=getattr(tvar,'calendar','standard'))
        try:
            tidx = pd.to_datetime(tvals)
        except Exception:
            tidx = pd.to_datetime([str(t) for t in tvals])
        lon_use = lon_arr.copy()
        if np.nanmax(lon_use) > 180:
            lon_use = ((lon_use + 180) % 360) - 180

        for name, (la0,la1,lo0,lo1) in SITE_BBOXES.items():
            mask = ((lat_arr>=la0)&(lat_arr<=la1)&
                    (lon_use>=lo0)&(lon_use<=lo1))
            idx  = np.flatnonzero(mask)
            if len(idx) < 5: continue
            area = nc.variables['area']
            fill = getattr(area,'_FillValue', None)
            total = np.zeros(len(tidx))
            cov   = np.zeros(len(tidx), dtype=int)
            for s,e in zip(*[
                    np.r_[idx[0], idx[np.where(np.diff(idx)!=1)[0]+1]],
                    np.r_[idx[np.where(np.diff(idx)!=1)[0]]+1, idx[-1]+1]]):
                blk = np.array(area[:,s:e], dtype=np.float32)
                valid = np.isfinite(blk)
                if fill is not None: valid &= (blk != fill)
                blk[~valid] = 0
                total += blk.sum(1)
                cov   += valid.sum(1).astype(int)
            ka = total.astype(float)
            ka[cov==0] = np.nan
            ka[cov/len(idx) < 0.2] = np.nan
            df_k = pd.DataFrame({'kelp_area':ka}, index=tidx)
            df_k['kelp_smooth'] = df_k['kelp_area'].rolling(
                4, center=True, min_periods=2).mean()
            raw_kelp[name] = df_k
    print(f'  ✓ Extracted {len(raw_kelp)} sites')
else:
    print(f'  ! NetCDF not found at {NC_PATH}')
    print('    Using synthetic data for demonstration.')

print(f'\n✓ Data loading complete')
print(f'  Kelp sites:  {len(raw_kelp)}')
print(f'  UI latbands: {sorted(ui_cache.keys())}')

Loading SST...
  ✓ SST loaded (OPeNDAP)
Loading upwelling...
  ✓ UI33
  ✓ UI36
  ✓ UI39
  ✓ UI42
Extracting kelp from NetCDF...


In [ ]:
# ============================================================
# CELL 3: FEATURE ENGINEERING
# Same pipeline as stress_test_v2 / loso_19sites
# ============================================================

def build_site_features(name, df_kelp, lat_mid, lon_mid):
    """De-seasonalize → EWS → SST → upwelling → labels."""
    df = df_kelp.copy()
    df.index = pd.to_datetime(df.index)

    # Quarterly align
    df.index = df.index.to_period('Q').to_timestamp(how='start')
    df = df.groupby(df.index).mean()    # average duplicates
    df = df.sort_index()

    # Deseasonalize
    df['q'] = df.index.quarter
    base = df.loc['1984':'2013'].dropna(subset=['kelp_smooth'])
    if len(base) < 8:
        return None
    med = base.groupby('q')['kelp_smooth'].median()
    mad = base.groupby('q')['kelp_smooth'].apply(
        lambda x: np.median(np.abs(x - np.median(x))) + 1e-9)
    df['kelp_q_z'] = (df['kelp_smooth'] - df['q'].map(med)) / df['q'].map(mad)
    df.drop(columns=['q'], inplace=True)

    # EWS
    z   = df['kelp_q_z']
    ar1 = z.rolling(ROLL_WIN, min_periods=ROLL_WIN//2).apply(
        lambda x: pd.Series(x).autocorr(lag=1), raw=True)
    var = z.rolling(ROLL_WIN, min_periods=ROLL_WIN//2).var()
    ar1_z = (ar1 - ar1.mean()) / (ar1.std() + 1e-9)
    var_z = (var - var.mean()) / (var.std() + 1e-9)
    df['ews_composite'] = (ar1_z + var_z) / 2

    # SST
    if sst_ca is not None:
        lon_360 = (lon_mid + 360) % 360
        sst_s   = sst_ca.sel({
            lat_nm: slice(lat_mid-0.8, lat_mid+0.8),
            lon_nm: slice(lon_360-0.8, lon_360+0.8)
        })
        if sst_s.sizes.get(lat_nm, 0) == 0:
            sst_s = sst_ca.sel({
                lat_nm: slice(lat_mid+0.8, lat_mid-0.8),
                lon_nm: slice(lon_360-0.8, lon_360+0.8)
            })
        sst_m = sst_s.mean(dim=[lat_nm, lon_nm], skipna=True).to_series()
        sst_m.index = pd.to_datetime(sst_m.index).tz_localize(None)
        clim    = sst_m.loc['1991':'2020'].groupby(
            sst_m.loc['1991':'2020'].index.month).mean()
        anom    = sst_m - sst_m.index.month.map(clim)
        sst_q   = anom.resample('QS').max()
        sst_q.index = sst_q.index.to_period('Q').to_timestamp(how='start')
        sst_q   = sst_q.reindex(df.index)
        df['heat_lag4'] = sst_q.shift(HEAT_LAG).values
    else:
        df['heat_lag4'] = np.nan

    # Upwelling
    if ui_cache:
        best_lat = min(ui_cache.keys(), key=lambda k: abs(k-lat_mid))
        ui       = ui_cache[best_lat]
        ui_q     = ui['ui_anom'].resample('QS').mean()
        ui_q.index = ui_q.index.to_period('Q').to_timestamp(how='start')
        ui_q     = ui_q.reindex(df.index)
        df['upwelling'] = ui_q.shift(1).values
    else:
        df['upwelling'] = np.nan

    df['heat_x_ews'] = df['heat_lag4'] * df['ews_composite']

    # Labels
    df['kelp_z_1yr'] = df['kelp_q_z'].rolling(4, min_periods=4).mean()
    base_z = df.loc['1984':'2013', 'kelp_z_1yr'].dropna()
    if len(base_z) < 4:
        return None
    thr = base_z.quantile(0.10)
    df['suppressed'] = (df['kelp_z_1yr'] <= thr).astype(int)
    s = df['suppressed']
    df['onset'] = ((s==1) & (s.shift(1)==0)).astype(int)

    return df


# ── Build feature dataframes for all sites ────────────────────
# If raw_kelp is empty (no NetCDF), create synthetic demo data
if not raw_kelp:
    print('Generating synthetic kelp data for demonstration...')
    np.random.seed(42)
    tidx = pd.date_range('1984-01-01', '2024-10-01', freq='QS')
    for row in SITES_DF.itertuples():
        t  = np.linspace(0, 40, len(tidx))
        # Synthetic signal: seasonal + trend + collapses
        signal  = (10000 + 1000*np.sin(2*np.pi*t/1)
                   - 50*t
                   + 800*np.random.randn(len(t)))
        signal[signal < 0] = 0
        raw_kelp[row.name] = pd.DataFrame({
            'kelp_area':   signal,
            'kelp_smooth': pd.Series(signal).rolling(
                4, center=True, min_periods=2).mean().values
        }, index=tidx)

print('Building features for each site...')
site_data = {}
for _, row in SITES_DF.iterrows():
    name = row['name']
    if name not in raw_kelp:
        print(f'  SKIP {name}: no kelp data')
        continue
    result = build_site_features(
        name, raw_kelp[name], row['lat'], row['lon'])
    if result is not None:
        site_data[name] = result
        n_onset = int(result['onset'].sum())
        print(f'  ✓ {name:22s}: {len(result)}q | onset={n_onset}')
    else:
        print(f'  SKIP {name}: insufficient data')

print(f'\n✓ {len(site_data)}/{len(SITES_DF)} sites ready')

In [ ]:
# ============================================================
# CELL 4: HELPERS
# ============================================================

def block_bootstrap_auc(score, y, block_len=4, B=BOOTSTRAP_B, seed=42):
    score = np.asarray(score, float)
    y     = np.asarray(y, int)
    n     = len(y)
    starts   = np.arange(0, n - block_len + 1)
    n_blocks = int(np.ceil(n / block_len))
    rng      = np.random.default_rng(seed)
    aucs     = []
    for _ in range(B):
        chosen = rng.choice(starts, size=n_blocks, replace=True)
        idx    = np.concatenate(
            [np.arange(s, s+block_len) for s in chosen])[:n]
        sb, yb = score[idx], y[idx]
        if np.unique(yb).size < 2: continue
        aucs.append(roc_auc_score(yb, sb))
    aucs = np.array(aucs)
    if len(aucs) < 50:
        return dict(auc=np.nan, ci_lo=np.nan, ci_hi=np.nan)
    return dict(auc=aucs.mean(),
                ci_lo=np.quantile(aucs, 0.025),
                ci_hi=np.quantile(aucs, 0.975))


def run_loro_fold(train_names, test_name, site_data):
    """Train on train_names, test on test_name.
    Returns dict with AUC, CI, prob, y, etc."""
    train_frames = [
        site_data[n][FEATURES + [TARGET]].dropna()
        for n in train_names
        if n in site_data
    ]
    if not train_frames:
        return None
    train_df = pd.concat(train_frames)
    test_df  = site_data[test_name][FEATURES + [TARGET]].dropna()

    if train_df[TARGET].sum() < 2:
        return None
    if test_df[TARGET].nunique() < 2:
        return dict(auc=np.nan, ci_lo=np.nan, ci_hi=np.nan,
                    prob=None, y=test_df[TARGET].values,
                    n_train=len(train_df), n_test=len(test_df),
                    n_onset=int(test_df[TARGET].sum()),
                    sig=False, skip_reason='single_class')

    scaler = StandardScaler()
    lr     = LogisticRegression(C=0.5, class_weight='balanced', max_iter=1000)
    lr.fit(scaler.fit_transform(train_df[FEATURES]),
           train_df[TARGET].astype(int).values)
    prob   = lr.predict_proba(
        scaler.transform(test_df[FEATURES]))[:, 1]
    y_test = test_df[TARGET].astype(int).values

    bb = block_bootstrap_auc(prob, y_test)
    return dict(
        auc    = bb['auc'],
        ci_lo  = bb['ci_lo'],
        ci_hi  = bb['ci_hi'],
        prob   = prob,
        y      = y_test,
        t_idx  = test_df.index,
        n_train= len(train_df),
        n_test = len(test_df),
        n_onset= int(y_test.sum()),
        sig    = (bb['ci_lo'] > 0.5
                  if not np.isnan(bb['ci_lo']) else False),
        coef   = dict(zip(FEATURES, lr.coef_[0]))
    )


def sig_stars(p):
    if pd.isna(p):   return ''
    if p < 0.001:    return '***'
    if p < 0.01:     return '**'
    if p < 0.05:     return '*'
    return '(ns)'

print('✓ Helpers ready')

In [ ]:
# ============================================================
# CELL 5: NOVEL INTRA-REGION LORO
#
# Three cross-validation strategies are compared:
#
#  A) Classic LORO       – train on ALL other regions, test 1 site
#  B) Intra-region LORO  – train on sites WITHIN same region, test 1
#  C) Cross-region LORO  – train on sites from OTHER regions, test 1
#
# Strategy B is the novel design: asks whether the EWS signal
# is learnable purely from within-regime dynamics, without any
# signal from different oceanographic regimes.
# ============================================================

results = []   # one row per (site, strategy)

site_names   = [r.name for r in SITES_DF.itertuples()
                if r.name in site_data]
site_region  = {r.name: r.region
                for r in SITES_DF.itertuples()}
site_lat     = {r.name: r.lat
                for r in SITES_DF.itertuples()}

print(f'Running 3-strategy LORO for {len(site_names)} sites...\n')

for test_name in site_names:
    region_test = site_region[test_name]
    lat_test    = site_lat[test_name]
    central     = (CENTRAL_LAT_MIN <= lat_test <= CENTRAL_LAT_MAX)

    within_region  = [n for n in site_names
                      if n != test_name
                      and site_region[n] == region_test]
    outside_region = [n for n in site_names
                      if n != test_name
                      and site_region[n] != region_test]
    all_other      = [n for n in site_names if n != test_name]

    print(f'  {test_name:22s} | {REGION_META[region_test]["label"]:15s}', end='')

    for strategy, train_pool in [
        ('A_classic',     all_other),
        ('B_intraregion', within_region),
        ('C_crossregion', outside_region),
    ]:
        if len(train_pool) < 1:
            print(f'  [{strategy}:SKIP]', end='')
            continue

        res = run_loro_fold(train_pool, test_name, site_data)
        if res is None:
            print(f'  [{strategy}:FAIL]', end='')
            continue

        results.append({
            'site':     test_name,
            'region':   region_test,
            'lat':      lat_test,
            'central':  central,
            'strategy': strategy,
            'n_train':  res['n_train'],
            'n_test':   res['n_test'],
            'n_onset':  res['n_onset'],
            'auc':      res['auc'],
            'ci_lo':    res['ci_lo'],
            'ci_hi':    res['ci_hi'],
            'sig':      res['sig'],
            'prob':     res.get('prob'),
            'y':        res.get('y'),
            't_idx':    res.get('t_idx'),
            'coef':     res.get('coef'),
        })

        flag = '✓' if res.get('sig') else ' '
        auc  = res['auc']
        print(f'  [{strategy}: {auc:.3f}{flag}]', end='')

    print()

res_df = pd.DataFrame(results)

print('\n' + '='*60)
print('SUMMARY')
print('='*60)
for strat, g in res_df.groupby('strategy'):
    valid = g.dropna(subset=['auc'])
    print(f'  {strat}: N={len(valid):2d}  '
          f'mean AUC={valid["auc"].mean():.3f}  '
          f'sig={valid["sig"].sum()}/{len(valid)}')

In [ ]:
# ============================================================
# CELL 6: FIG 1 — TRIPLE FOREST PLOT
# Each site shown three times (one dot per strategy)
# Side-by-side panels: Classic / Intra-region / Cross-region
# ============================================================

strategies = [
    ('A_classic',     'A. Classic LORO\n(train on all other sites)',     '#1a3a5c'),
    ('B_intraregion', 'B. Intra-Region LORO (NOVEL)\n(train within same region only)', '#8b1a1a'),
    ('C_crossregion', 'C. Cross-Region LORO\n(train on different regions only)',       '#1a5c2a'),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 11), sharey=True)
fig.patch.set_facecolor('#f8f9fb')
fig.suptitle(
    'Fig. 1 — Triple Cross-Validation Strategy Comparison\n'
    'All 15 sites × 3 training regimes | Filled = significant (CI > 0.5)',
    fontsize=14, fontweight='bold', y=1.01, color='#1a1a2e'
)

# Sort sites north→south
sorted_sites = sorted([n for n in site_names if n in site_data],
                      key=lambda n: -site_lat[n])

for ax, (strat, title, hdr_col) in zip(axes, strategies):
    sub = res_df[res_df['strategy'] == strat].set_index('site')
    ax.set_facecolor('#ffffff')

    # Central CA band
    central_idxs = [i for i, n in enumerate(sorted_sites)
                    if CENTRAL_LAT_MIN <= site_lat[n] <= CENTRAL_LAT_MAX]
    if central_idxs:
        ax.axhspan(min(central_idxs) - 0.5, max(central_idxs) + 0.5,
                   alpha=0.08, color='#d4622a', zorder=0)

    for i, name in enumerate(sorted_sites):
        if name not in sub.index:
            continue
        row   = sub.loc[name]
        col   = REGION_META[site_region[name]]['color']
        is_sig = row['sig']
        auc    = row['auc']
        ci_lo  = row['ci_lo']
        ci_hi  = row['ci_hi']
        alpha  = 1.0 if is_sig else 0.45

        if pd.isna(auc):
            ax.scatter([0.5], [i], marker='x', s=80,
                       color='#aaa', alpha=0.5, zorder=3)
            continue

        ax.errorbar(
            auc, i,
            xerr=[[max(auc - ci_lo, 0)], [max(ci_hi - auc, 0)]],
            fmt='o', color=col,
            markersize=9 if is_sig else 7,
            capsize=4, ecolor=col, alpha=alpha, lw=1.8,
            markerfacecolor=col if is_sig else 'white',
            markeredgewidth=2, zorder=4
        )

    ax.axvline(0.5, color='#888', lw=1.5, ls='--', alpha=0.6,
               label='Random (0.5)')
    ax.set_xlim(0.05, 1.15)
    ax.set_xlabel('AUC (95% CI, block bootstrap)', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold',
                 color=hdr_col, pad=10)

    # Label sig count
    sub_v = sub.dropna(subset=['auc'])
    n_sig = int(sub_v['sig'].sum())
    n_tot = len(sub_v)
    ax.text(0.97, 0.02,
            f'{n_sig}/{n_tot} sig\nmean={sub_v["auc"].mean():.3f}',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=10, fontweight='bold',
            color='#27ae60' if n_sig > 0 else '#888')

# Y-axis labels on left panel only
axes[0].set_yticks(range(len(sorted_sites)))
axes[0].set_yticklabels(
    [f"{n}  ({site_lat[n]:.1f}°N)"
     for n in sorted_sites],
    fontsize=9
)
for ax in axes[1:]:
    ax.set_yticks([])

# Region legend
legend_elements = [
    Line2D([0],[0], marker='o', color=REGION_META[r]['color'],
           lw=0, markersize=9,
           label=REGION_META[r]['label'])
    for r in ['norcal','midcal','bigsur','socal']
] + [
    mpatches.Patch(facecolor='#d4622a', alpha=0.12,
                   label='Central CA upwelling core'),
    Line2D([0],[0], marker='o', color='#555', lw=0, markersize=9,
           markerfacecolor='white', markeredgewidth=2,
           label='Not significant (open)'),
]
axes[1].legend(handles=legend_elements, fontsize=9,
               loc='lower right', framealpha=0.9)

fig.tight_layout()
out = FIG_DIR / 'fig1_triple_forest_plot.png'
fig.savefig(out, dpi=200, bbox_inches='tight', facecolor='#f8f9fb')
plt.show()
print(f'Saved: {out}')

In [ ]:
# ============================================================
# CELL 7: FIG 2 — AUC DELTA: INTRA vs CROSS vs CLASSIC
# Two panels:
#   A) Violin + box showing AUC distribution per strategy
#   B) Paired scatter: each site's intra vs cross AUC
#      (above diagonal = intra > cross, i.e. within-regime signal stronger)
# ============================================================

strat_labels = {
    'A_classic':     'Classic LORO',
    'B_intraregion': 'Intra-Region\n(Novel)',
    'C_crossregion': 'Cross-Region',
}
strat_colors = {
    'A_classic':     '#1a3a5c',
    'B_intraregion': '#c0392b',
    'C_crossregion': '#27ae60',
}

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(15, 6.5))
fig.patch.set_facecolor('#f8f9fb')

# ──── Panel A: Distribution per strategy ────────────────────
strat_aucs = {}
for strat in ['A_classic','B_intraregion','C_crossregion']:
    vals = res_df[res_df['strategy']==strat]['auc'].dropna().values
    strat_aucs[strat] = vals

positions = [1, 2, 3]
groups    = [strat_aucs[s] for s in
             ['A_classic','B_intraregion','C_crossregion']]
cols      = [strat_colors[s] for s in
             ['A_classic','B_intraregion','C_crossregion']]

vp = ax_a.violinplot(groups, positions=positions,
                     showmedians=False, showextrema=False)
for body, c in zip(vp['bodies'], cols):
    body.set_facecolor(c)
    body.set_alpha(0.25)

bp = ax_a.boxplot(groups, positions=positions, widths=0.38,
                   patch_artist=True,
                   medianprops=dict(color='white', lw=2.5),
                   whiskerprops=dict(lw=1.8),
                   capprops=dict(lw=1.8),
                   flierprops=dict(marker='o', markersize=5,
                                  alpha=0.5))
for patch, c in zip(bp['boxes'], cols):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)

rng = np.random.default_rng(0)
for pos, grp, c in zip(positions, groups, cols):
    jitter = rng.uniform(-0.09, 0.09, len(grp))
    ax_a.scatter(np.full(len(grp), pos) + jitter, grp,
                 color=c, s=55, alpha=0.85, zorder=5,
                 edgecolors='white', linewidths=0.8)

# Kruskal-Wallis test
if all(len(g) > 2 for g in groups):
    H, p_kw = kruskal(*groups)
    ax_a.text(0.97, 0.97,
              f'Kruskal-Wallis\np = {p_kw:.4f} {sig_stars(p_kw)}',
              transform=ax_a.transAxes, ha='right', va='top',
              fontsize=10,
              color='#27ae60' if p_kw < 0.05 else '#888')

# Pairwise: intra vs cross
if len(strat_aucs['B_intraregion']) > 2 and len(strat_aucs['C_crossregion']) > 2:
    u_bc, p_bc = mannwhitneyu(
        strat_aucs['B_intraregion'],
        strat_aucs['C_crossregion'],
        alternative='two-sided')
    ymax = max(max(g) for g in groups if len(g) > 0) + 0.06
    ax_a.plot([2, 3], [ymax, ymax], 'k-', lw=1.5)
    ax_a.text(2.5, ymax+0.01,
              f'Intra vs Cross\np={p_bc:.4f} {sig_stars(p_bc)}',
              ha='center', fontsize=9)

ax_a.axhline(0.5, ls='--', color='#888', lw=1.3, alpha=0.6)
ax_a.set_xticks(positions)
ax_a.set_xticklabels(
    [strat_labels[s] for s in
     ['A_classic','B_intraregion','C_crossregion']], fontsize=10)
ax_a.set_ylabel('AUC (block bootstrap)')
ax_a.set_title(
    'Fig. 2A — AUC Distribution by Strategy\n'
    'Does within-regime training compete with classic LORO?',
    fontsize=11)

# ──── Panel B: Paired scatter intra vs cross ─────────────────
# Pivot: one row per site
pivot = res_df.pivot_table(
    index='site', columns='strategy', values='auc')
pivot = pivot.dropna(subset=['B_intraregion','C_crossregion'])
pivot = pivot.join(
    pd.Series({n: site_region[n] for n in pivot.index}, name='region'))

lim = (0.25, 1.05)
ax_b.plot(lim, lim, '--', color='#888', lw=1.5, alpha=0.6,
          label='Equal AUC (diagonal)')
ax_b.axhline(0.5, ls=':', color='#888', lw=1.0, alpha=0.4)
ax_b.axvline(0.5, ls=':', color='#888', lw=1.0, alpha=0.4)

for site, row in pivot.iterrows():
    col = REGION_META.get(row['region'], {}).get('color', '#888')
    auc_cross = row['C_crossregion']
    auc_intra = row['B_intraregion']
    ax_b.scatter(auc_cross, auc_intra, color=col, s=80,
                 alpha=0.85, edgecolors='white', linewidths=0.8,
                 zorder=4)
    short = site.replace(' ', '\n')
    ax_b.annotate(short,
                  (auc_cross, auc_intra),
                  textcoords='offset points',
                  xytext=(5, 3),
                  fontsize=7.5, color='#333', alpha=0.85)

# Shade region: above diagonal = intra wins
ax_b.fill_between(lim, lim, [lim[1], lim[1]],
                  alpha=0.05, color='#c0392b',
                  label='Intra-region AUC higher')
ax_b.fill_between(lim, [lim[0], lim[0]], lim,
                  alpha=0.05, color='#27ae60',
                  label='Cross-region AUC higher')

n_intra_wins = int((pivot['B_intraregion'] > pivot['C_crossregion']).sum())
ax_b.text(0.97, 0.02,
          f'Intra > Cross: {n_intra_wins}/{len(pivot)} sites',
          transform=ax_b.transAxes, ha='right', va='bottom',
          fontsize=10, fontweight='bold')

ax_b.set_xlim(lim)
ax_b.set_ylim(lim)
ax_b.set_xlabel('Cross-Region AUC')
ax_b.set_ylabel('Intra-Region AUC (Novel)')
ax_b.set_title(
    'Fig. 2B — Intra vs Cross-Region AUC per Site\n'
    'Above diagonal: within-regime signal alone is predictive',
    fontsize=11)
ax_b.legend(fontsize=8.5, loc='upper left')

fig.tight_layout()
out = FIG_DIR / 'fig2_strategy_comparison.png'
fig.savefig(out, dpi=200, bbox_inches='tight', facecolor='#f8f9fb')
plt.show()
print(f'Saved: {out}')

In [ ]:
# ============================================================
# CELL 8: FIG 3 — INTRA-REGION AUC BY REGION
# Four panels, one per macro-region.
# Each panel: forest plot of within-region LORO folds
# + region-level mean AUC annotation
# ============================================================

intra = res_df[res_df['strategy'] == 'B_intraregion'].copy()
regions_ordered = ['norcal', 'midcal', 'bigsur', 'socal']

fig, axes = plt.subplots(1, 4, figsize=(18, 7),
                          gridspec_kw={'wspace': 0.45})
fig.patch.set_facecolor('#f8f9fb')
fig.suptitle(
    'Fig. 3 — Intra-Region LORO: Per-Region Breakdown\n'
    'Train on N−1 sites within same region, test on 1 held-out site',
    fontsize=13, fontweight='bold', color='#1a1a2e', y=1.02
)

for ax, region in zip(axes, regions_ordered):
    sub   = intra[intra['region'] == region].sort_values(
        'lat', ascending=False).reset_index(drop=True)
    col   = REGION_META[region]['color']
    label = REGION_META[region]['label']

    ax.set_facecolor('#ffffff')

    # Background stripe for central CA
    if region == 'midcal':
        ax.axvspan(0.0, 1.3, alpha=0.06, color=col, zorder=0)

    n_sig = 0
    for i, row in sub.iterrows():
        auc   = row['auc']
        ci_lo = row['ci_lo']
        ci_hi = row['ci_hi']
        is_sig = row['sig']
        if is_sig: n_sig += 1

        name_short = row['site'].replace(' ', '\n')

        if pd.isna(auc):
            ax.text(0.5, i, 'N/A', ha='center', va='center',
                    color='#aaa', fontsize=9)
            continue

        ax.errorbar(
            auc, i,
            xerr=[[max(auc-ci_lo,0)],[max(ci_hi-auc,0)]],
            fmt='o', color=col,
            markersize=10 if is_sig else 8,
            capsize=5, ecolor=col, alpha=1.0 if is_sig else 0.45,
            lw=2,
            markerfacecolor=col if is_sig else 'white',
            markeredgewidth=2.2, zorder=4
        )

        # Sig annotation
        if is_sig:
            ax.text(ci_hi + 0.04, i, '★',
                    color='#f39c12', fontsize=12, va='center')

    ax.axvline(0.5, ls='--', color='#888', lw=1.4, alpha=0.6)

    valid_aucs = sub['auc'].dropna()
    mean_auc   = valid_aucs.mean()
    ax.axvline(mean_auc, ls='-', color=col, lw=2.0, alpha=0.5,
               label=f'Mean AUC = {mean_auc:.3f}')

    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels(
        [f"{r['site']}\n({r['lat']:.1f}°N)"
         for _, r in sub.iterrows()],
        fontsize=8.5
    )
    ax.set_xlim(0.05, 1.20)
    ax.set_xlabel('Intra-Region AUC')

    n_valid = int(sub['auc'].notna().sum())
    ax.set_title(
        f'{label}\nMean AUC = {mean_auc:.3f}\n'
        f'{n_sig}/{n_valid} significant',
        fontsize=10.5, fontweight='bold', color=col, pad=10
    )

    # N train annotation
    ax.text(0.97, -0.08,
            f'Training: {int(sub["n_train"].mean())} qtrs avg',
            transform=ax.transAxes, ha='right',
            fontsize=8, color='#666', style='italic')

fig.tight_layout()
out = FIG_DIR / 'fig3_intraregion_by_region.png'
fig.savefig(out, dpi=200, bbox_inches='tight', facecolor='#f8f9fb')
plt.show()
print(f'Saved: {out}')

In [ ]:
# ============================================================
# CELL 9: FIG 4 — RISK TIMELINE GALLERY
# Best site from each region under intra-region LORO
# Shows what the model actually "sees" within a regime
# ============================================================

intra_valid = intra.dropna(subset=['auc'])

# Pick best site per region
best_per_region = (
    intra_valid.sort_values('auc', ascending=False)
               .groupby('region').first()
               .reset_index()
)

n_panels = len(best_per_region)
fig, axes = plt.subplots(n_panels, 1, figsize=(15, 4.5*n_panels))
if n_panels == 1: axes = [axes]
fig.patch.set_facecolor('#f8f9fb')
fig.suptitle(
    'Fig. 4 — Risk Score Timelines: Best Site per Region\n'
    'Model trained purely on within-region sites (intra-region LORO)',
    fontsize=13, fontweight='bold', y=1.01, color='#1a1a2e'
)

for ax, (_, brow) in zip(axes, best_per_region.iterrows()):
    site   = brow['site']
    region = brow['region']
    col    = REGION_META[region]['color']

    if site not in site_data:
        ax.text(0.5, 0.5, f'{site} — data not available',
                ha='center', transform=ax.transAxes)
        continue

    # Re-run the fold to get time-indexed risk
    region_sites = [n for n in site_names if site_region[n] == region]
    train_names  = [n for n in region_sites if n != site]
    res_fold     = run_loro_fold(train_names, site, site_data)

    if res_fold is None or res_fold.get('prob') is None:
        ax.text(0.5, 0.5, f'{site} — insufficient data',
                ha='center', transform=ax.transAxes)
        continue

    t_idx = res_fold['t_idx']
    risk  = pd.Series(res_fold['prob'], index=t_idx)
    y_vec = pd.Series(res_fold['y'], index=t_idx)

    df_site = site_data[site]
    supp    = df_site['suppressed'].reindex(t_idx).fillna(0).astype(int)
    onset   = df_site['onset'].reindex(t_idx).fillna(0).astype(int)
    kelp_c  = 'kelp_smooth' if 'kelp_smooth' in df_site.columns else 'kelp_area'
    kelp    = df_site[kelp_c].reindex(t_idx)
    kelp_n  = (kelp - kelp.min()) / (kelp.max() - kelp.min() + 1e-9)

    ax.set_facecolor('#ffffff')

    # Suppression shading
    in_supp = False
    for j, (t, sv) in enumerate(supp.items()):
        if sv == 1 and not in_supp:
            t_start = t; in_supp = True
        elif sv == 0 and in_supp:
            ax.axvspan(t_start, t, alpha=0.1, color='#e74c3c', zorder=0)
            in_supp = False
    if in_supp:
        ax.axvspan(t_start, supp.index[-1], alpha=0.1,
                   color='#e74c3c', zorder=0)

    # EWS composite overlay
    ews = df_site['ews_composite'].reindex(t_idx)
    ews_n = (ews - ews.min()) / (ews.max() - ews.min() + 1e-9)
    ax.fill_between(ews_n.index, 0, ews_n*0.4,
                    alpha=0.12, color='#8e44ad', zorder=1,
                    label='EWS composite (scaled)')

    ax.plot(kelp_n.index, kelp_n, color='#27ae60', alpha=0.4,
            lw=1.8, label='Kelp (normalized)')
    ax.plot(risk.index, risk, color=col, lw=2.5,
            label='Intra-region LORO risk score', zorder=3)
    ax.axhline(THRESHOLD, ls='--', color='#e74c3c', lw=1.8, alpha=0.9,
               label=f'Alert threshold ({THRESHOLD})')

    # Onset markers
    onset_locs = risk[onset == 1]
    ax.scatter(onset_locs.index, onset_locs.values,
               marker='v', s=200, color='#e74c3c', zorder=6,
               edgecolors='white', linewidths=1.2,
               label='Actual collapse onset')

    # Count warnings
    onset_idx = np.where(onset.values == 1)[0]
    warned = sum(
        1 for k in onset_idx
        if (risk.values[max(0,k-WARN_WINDOW):k] >= THRESHOLD).any()
    )

    auc_val = brow['auc']
    ax.set_title(
        f'{site}  |  {REGION_META[region]["label"]}  |  '
        f'Intra-Region LORO  |  '
        f'AUC = {auc_val:.3f}  '
        f'CI=[{brow["ci_lo"]:.3f}, {brow["ci_hi"]:.3f}]  |  '
        f'{warned}/{int(onset.sum())} collapses warned within {WARN_WINDOW}q  |  '
        f'Train sites: {", ".join(train_names)}',
        fontsize=9.5, color=col, pad=7
    )
    ax.set_ylabel('Risk / Normalized Kelp')
    ax.set_ylim(-0.05, 1.22)
    ax.legend(fontsize=8.5, ncol=3, loc='upper left', framealpha=0.9)

axes[-1].set_xlabel('Year')
fig.tight_layout()
out = FIG_DIR / 'fig4_risk_timelines.png'
fig.savefig(out, dpi=200, bbox_inches='tight', facecolor='#f8f9fb')
plt.show()
print(f'Saved: {out}')

In [ ]:
# ============================================================
# CELL 10: FIG 5 — FEATURE IMPORTANCE HEATMAP
# Average logistic regression coefficients (intra-region)
# grouped by region — reveals which features matter WITHIN
# each oceanographic regime
# ============================================================

# Collect coefficients from intra-region folds
coef_rows = []
for _, row in intra_valid.iterrows():
    if row['coef'] is None: continue
    for feat, val in row['coef'].items():
        coef_rows.append({
            'site':   row['site'],
            'region': row['region'],
            'feat':   feat,
            'coef':   val
        })
coef_df = pd.DataFrame(coef_rows)

if len(coef_df) == 0:
    print('No coefficient data available — skipping Fig 5')
else:
    pivot_coef = coef_df.groupby(['region','feat'])['coef'].mean().unstack()
    # Reorder columns by typical importance
    col_order = [f for f in FEATURES if f in pivot_coef.columns]
    pivot_coef = pivot_coef[col_order]

    feat_labels = {
        'ews_composite': 'EWS Composite\n(CSD signal)',
        'heat_lag4':     'SST Heat Stress\n(4q lag)',
        'upwelling':     'Upwelling Anomaly\n(1q lag)',
        'heat_x_ews':    'Heat × EWS\n(interaction)',
    }
    pivot_coef.columns = [feat_labels.get(c, c) for c in pivot_coef.columns]
    pivot_coef.index   = [REGION_META[r]['label'] for r in pivot_coef.index]

    fig, (ax_heat, ax_bar) = plt.subplots(
        1, 2, figsize=(15, 5),
        gridspec_kw={'width_ratios': [2, 1]})
    fig.patch.set_facecolor('#f8f9fb')

    # ─── Heatmap ───────────────────────────────────────────
    vmax = pivot_coef.abs().max().max()
    cmap = LinearSegmentedColormap.from_list(
        'rdbu', ['#c0392b', '#ffffff', '#1a5276'])

    im = ax_heat.imshow(
        pivot_coef.values, cmap=cmap,
        vmin=-vmax, vmax=vmax, aspect='auto')

    ax_heat.set_xticks(range(len(pivot_coef.columns)))
    ax_heat.set_xticklabels(pivot_coef.columns, fontsize=9.5,
                             ha='center')
    ax_heat.set_yticks(range(len(pivot_coef.index)))
    ax_heat.set_yticklabels(pivot_coef.index, fontsize=10)

    for ri in range(len(pivot_coef.index)):
        for ci in range(len(pivot_coef.columns)):
            val = pivot_coef.values[ri, ci]
            txt = f'{val:.2f}'
            color = 'white' if abs(val) > vmax*0.5 else '#222'
            ax_heat.text(ci, ri, txt, ha='center', va='center',
                         fontsize=10, fontweight='bold', color=color)

    plt.colorbar(im, ax=ax_heat, label='Mean LR coefficient',
                 fraction=0.035)
    ax_heat.set_title(
        'Fig. 5A — Feature Importance by Region\n'
        'Mean logistic regression coefficient (intra-region LORO)',
        fontsize=11)

    # ─── Bar: EWS vs SST contribution by region ────────────
    ews_col = feat_labels.get('ews_composite','EWS Composite\n(CSD signal)')
    sst_col = feat_labels.get('heat_lag4','SST Heat Stress\n(4q lag)')
    if ews_col in pivot_coef.columns and sst_col in pivot_coef.columns:
        x = np.arange(len(pivot_coef.index))
        w = 0.35
        ax_bar.barh(x + w/2, pivot_coef[ews_col],
                    height=w, color='#8b1a1a', alpha=0.8,
                    label='EWS (CSD)')
        ax_bar.barh(x - w/2, pivot_coef[sst_col],
                    height=w, color='#1a3a5c', alpha=0.8,
                    label='SST Heat Stress')
        ax_bar.axvline(0, color='#888', lw=1.2)
        ax_bar.set_yticks(x)
        ax_bar.set_yticklabels(pivot_coef.index, fontsize=10)
        ax_bar.set_xlabel('Mean coefficient')
        ax_bar.set_title(
            'Fig. 5B — EWS vs SST\nby Region',
            fontsize=11)
        ax_bar.legend(fontsize=9)
    else:
        ax_bar.text(0.5, 0.5, 'N/A', ha='center',
                    transform=ax_bar.transAxes)

    fig.tight_layout()
    out = FIG_DIR / 'fig5_feature_importance.png'
    fig.savefig(out, dpi=200, bbox_inches='tight', facecolor='#f8f9fb')
    plt.show()
    print(f'Saved: {out}')

In [ ]:
# ============================================================
# CELL 11: SUMMARY TABLE + STATS PRINTOUT
# ============================================================

print('=' * 68)
print('  INTRA-REGION LORO — COMPLETE STATS SUMMARY')
print('=' * 68)

# Per-region summary
for region in regions_ordered:
    sub = intra[intra['region'] == region].dropna(subset=['auc'])
    col = REGION_META[region]['label']
    print(f'\n  {col}:')
    print(f'    Sites:      {len(sub)}')
    print(f'    Mean AUC:   {sub["auc"].mean():.3f}')
    print(f'    Sig sites:  {int(sub["sig"].sum())}/{len(sub)}')
    for _, r in sub.sort_values('auc', ascending=False).iterrows():
        flag = '✓ sig' if r['sig'] else '     '
        ci = (f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]"
              if not pd.isna(r['ci_lo']) else '[N/A]')
        print(f'    {flag}  {r["site"]:22s}  AUC={r["auc"]:.3f}  '
              f'CI={ci}  n_onset={r["n_onset"]}')

# Cross-strategy
print('\n' + '=' * 68)
print('  STRATEGY COMPARISON')
print('=' * 68)
for strat, label in strat_labels.items():
    sub = res_df[res_df['strategy'] == strat].dropna(subset=['auc'])
    print(f'  {label:25s}: N={len(sub):2d}  '
          f'mean={sub["auc"].mean():.3f}  '
          f'sig={int(sub["sig"].sum())}/{len(sub)}')

# Central CA analysis (intra only)
central_intra = intra[intra['central'] == True].dropna(subset=['auc'])
noncent_intra = intra[intra['central'] == False].dropna(subset=['auc'])
print('\n' + '=' * 68)
print('  CENTRAL CA vs NON-CENTRAL (intra-region strategy)')
print('=' * 68)
print(f'  Central mean AUC:     {central_intra["auc"].mean():.3f}  '
      f'(N={len(central_intra)})')
print(f'  Non-central mean AUC: {noncent_intra["auc"].mean():.3f}  '
      f'(N={len(noncent_intra)})')
if len(central_intra) > 1 and len(noncent_intra) > 1:
    u, p = mannwhitneyu(central_intra['auc'], noncent_intra['auc'],
                        alternative='greater')
    print(f'  Mann-Whitney U p:     {p:.4f} {sig_stars(p)}')

print('\n' + '=' * 68)
print('  KEY NOVELTY STATEMENT')
print('=' * 68)
intra_valid2 = intra.dropna(subset=['auc'])
n_sig_intra  = int(intra_valid2['sig'].sum())
n_tot_intra  = len(intra_valid2)
print(f'''
  This notebook introduces INTRA-REGION LORO cross-validation,
  a novel evaluation design that asks: can an EWS model trained
  ONLY on sites within the same oceanographic regime generalize
  to an unseen site in that regime?

  Result: {n_sig_intra}/{n_tot_intra} sites achieve statistically
  significant AUC (CI > 0.5) under this strict constraint.

  Implication: The EWS signal is learnable from within-regime
  dynamics alone — not merely from cross-regime contrast.
  This strengthens the mechanistic interpretation that CSD-based
  EWS captures genuine ecosystem-level instability rather than
  an artifact of geography.
''')

figs = sorted(FIG_DIR.glob('*.png'))
print(f'All outputs in: {FIG_DIR.resolve()}')
for f in figs:
    print(f'  {f.name}')